In [ ]:
pip install kafka-python

In [ ]:
import json
import time
import yfinance as yf
from kafka import KafkaProducer
from datetime import datetime

In [ ]:
KAFKA_BROKER = "Add your IP address:9092"
TOPIC = "demo_test"
stocks = ["AAPL", "TSLA", "GOOG", "AMZN"]

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)
print("Real Stock Producer Started...")
serial_no = 1   # 🔹 serial counter

In [ ]:
try:
    while True:
        for symbol in stocks:
            ticker = yf.Ticker(symbol)
            price = ticker.history(period="1d", interval="1m")["Close"].iloc[-1]

            data = {
                "serial_no": serial_no,
                "file_name": f"stock_data_{serial_no}.json",
                "symbol": symbol,
                "price": float(price),
                "timestamp": datetime.now().isoformat()
            }

            producer.send(TOPIC, value=data)
            print("Sent:", data)

            serial_no += 1   # 🔹 increase serial

        producer.flush()
        time.sleep(1)

except KeyboardInterrupt:
    print("Stopping Producer...")
finally:
    producer.close()